[Reference](https://medium.com/@harishk3493/supercharge-your-images-with-ai-a-simple-real-esrgan-image-enhancer-in-python-6067d343d01f)

# Setup & Dependencies

```
torchvision
torchaudio
pytorch-lightning>=2.2.0
ultralytics
basicsr
realesrgan
```

In [1]:
import os
import cv2
import torch
import numpy as np
from PIL import Image, ImageEnhance
from basicsr.archs.rrdbnet_arch import RRDBNet
from basicsr.utils.download_util import load_file_from_url
from realesrgan import RealESRGANer

def enhance_single_image(
    input_path,
    output_path,
    model_name='RealESRGAN_x4plus',
    scale=4,
    tile_size=512,
    brightness=1.0,
    contrast=1.0,
    saturation=1.0,
    use_cuda=True
):
    # Set device
    device = torch.device('cuda' if use_cuda and torch.cuda.is_available() else 'cpu')
    half = device.type == 'cuda'

    # Download model if not present
    model_path = os.path.join('weights', f'{model_name}.pth')
    os.makedirs('weights', exist_ok=True)

    if not os.path.exists(model_path) or os.path.getsize(model_path) < 1000:
        url = {
            'RealESRGAN_x4plus': 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth'
        }.get(model_name)
        if not url:
            raise ValueError(f"No download URL for model: {model_name}")
        load_file_from_url(url, model_dir='weights')

    # Load model
    model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=scale)
    upsampler = RealESRGANer(
        scale=scale,
        model_path=model_path,
        model=model,
        tile=tile_size,
        tile_pad=32,
        pre_pad=0,
        half=half,
        device=device
    )

    # Read and convert image
    img = cv2.imread(input_path, cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(f"Cannot read image: {input_path}")
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Enhance
    with torch.no_grad():
        output_rgb, _ = upsampler.enhance(img_rgb, outscale=scale)
    output_bgr = cv2.cvtColor(output_rgb, cv2.COLOR_RGB2BGR)

    # Apply enhancements using PIL
    img_pil = Image.fromarray(cv2.cvtColor(output_bgr, cv2.COLOR_BGR2RGB))
    if brightness != 1.0:
        img_pil = ImageEnhance.Brightness(img_pil).enhance(brightness)
    if contrast != 1.0:
        img_pil = ImageEnhance.Contrast(img_pil).enhance(contrast)
    if saturation != 1.0:
        img_pil = ImageEnhance.Color(img_pil).enhance(saturation)

    # Save
    final_output = cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2BGR)
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    cv2.imwrite(output_path, final_output)
    print(f"Enhanced image saved to: {output_path}")


# Example usage:
if __name__ == "__main__":
    enhance_single_image(
        input_path='input.png',
        output_path='output.png',
        model_name='RealESRGAN_x4plus',
        scale=4,
        tile_size=512,
        brightness=1.0,
        contrast=1.0,
        saturation=1.0,
        use_cuda=True
    )